# Phase 3 — Verifikasi artifact TF-IDF berkelompok
Notebook menggunakan artifact tersimpan, tanpa fitting. Fold pertama StratifiedGroupKFold (5 fold, shuffle, seed 42) menempatkan clean_text identik di satu split.


In [1]:
from pathlib import Path
import sys
import json
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'PRD.md').exists():
    ROOT = ROOT.parent
if not (ROOT / 'PRD.md').exists():
    raise RuntimeError('Buka notebook dari root proyek atau direktori notebooks')
sys.path.insert(0, str(ROOT))
from src.audit import describe_dataset


In [2]:
manifest = json.loads((ROOT / 'data/processed/grouped_v1/manifest.json').read_text(encoding='utf-8'))
print(json.dumps({key: manifest[key] for key in ['experiment', 'cleaning_version', 'split', 'tfidf', 'environment']}, indent=2))


{
  "experiment": "grouped_v1",
  "cleaning_version": "legacy-v1",
  "split": {
    "method": "StratifiedGroupKFold",
    "group": "clean_text",
    "n_splits": 5,
    "selected_fold": 0,
    "shuffle": true,
    "random_state": 42,
    "requested_test_size": 0.2,
    "actual_test_size": 0.19999599847942218
  },
  "tfidf": {
    "max_features": 20000,
    "ngram_range": [
      1,
      2
    ],
    "min_df": 2,
    "max_df": 0.95,
    "sublinear_tf": true,
    "stop_words": null
  },
  "environment": {
    "python": "3.13.0",
    "packages": {
      "numpy": "2.5.5",
      "pandas": "3.1.1",
      "scipy": "1.18.1",
      "scikit-learn": "1.9.1",
      "joblib": "1.6.0"
    }
  }
}


In [3]:
import os
from src.artifacts import validate_artifacts
previous_cwd = Path.cwd()
try:
    os.chdir(ROOT)  # manifest menyimpan path relatif terhadap root proyek
    validation = validate_artifacts('data/processed/processed_reviews.csv',
                                    'data/processed/grouped_v1', 'models/grouped_v1')
finally:
    os.chdir(previous_cwd)
print(json.dumps(validation, indent=2))


{
  "train": {
    "shape": [
      39985,
      20000
    ],
    "zero_vectors": 117,
    "class_counts": {
      "Negative": 20242,
      "Positive": 16740,
      "Neutral": 3003
    }
  },
  "test": {
    "shape": [
      9996,
      20000
    ],
    "zero_vectors": 26,
    "class_counts": {
      "Negative": 5060,
      "Positive": 4185,
      "Neutral": 751
    }
  },
  "overlap_clean_text": 0
}


## Validasi dan keterbatasan
Validasi memeriksa checksum, cakupan baris, urutan label, hasil transformasi, IDF training, dan overlap teks. Transform hanya untuk verifikasi di memori. Vektor kosong dipertahankan; diagnosis ada di reports/zero_vectors.csv. Vectorizer lama dan seluruh artifact baseline tetap tersedia.
